# 视觉-语言模型高级教程

本教程涵盖高级主题：多模态融合、检索系统、性能优化等。

## 目录
1. [多模态融合策略](#1-多模态融合策略)
2. [图文检索系统实现](#2-图文检索系统实现)
3. [模型优化技术](#3-模型优化技术)
4. [训练技巧](#4-训练技巧)
5. [评估指标](#5-评估指标)

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)

## 1. 多模态融合策略

### 1.1 早期融合 (Early Fusion)

In [ ]:
class EarlyFusion(nn.Module):
    """早期融合：在特征提取阶段融合多模态信息"""
    def __init__(self, image_dim=512, text_dim=512, hidden_dim=512):
        super().__init__()
        self.image_proj = nn.Linear(image_dim, hidden_dim)
        self.text_proj = nn.Linear(text_dim, hidden_dim)
        self.fusion_layers = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(hidden_dim, nhead=8, batch_first=True),
            num_layers=4
        )
        self.output_proj = nn.Linear(hidden_dim, hidden_dim)
    
    def forward(self, image_features, text_features):
        img = self.image_proj(image_features)
        txt = self.text_proj(text_features)
        combined = torch.cat([img, txt], dim=1)
        fused = self.fusion_layers(combined)
        return self.output_proj(fused.mean(dim=1))

# 测试
early_fusion = EarlyFusion().to(device)
img_feat = torch.randn(4, 49, 512).to(device)  # 7x7 patches
txt_feat = torch.randn(4, 32, 512).to(device)  # 32 tokens
output = early_fusion(img_feat, txt_feat)
print(f'早期融合输出: {output.shape}')

### 1.2 晚期融合 (Late Fusion)

In [ ]:
class LateFusion(nn.Module):
    """晚期融合：各模态独立编码后再融合"""
    def __init__(self, image_dim=512, text_dim=512, output_dim=256):
        super().__init__()
        self.image_encoder = nn.Sequential(
            nn.Linear(image_dim, 512), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(512, output_dim)
        )
        self.text_encoder = nn.Sequential(
            nn.Linear(text_dim, 512), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(512, output_dim)
        )
        self.fusion = nn.Sequential(
            nn.Linear(output_dim * 2, output_dim),
            nn.ReLU(),
            nn.Linear(output_dim, output_dim)
        )
    
    def forward(self, image_features, text_features):
        img_emb = self.image_encoder(image_features)
        txt_emb = self.text_encoder(text_features)
        combined = torch.cat([img_emb, txt_emb], dim=-1)
        return self.fusion(combined)

# 测试
late_fusion = LateFusion().to(device)
img_feat = torch.randn(4, 512).to(device)
txt_feat = torch.randn(4, 512).to(device)
output = late_fusion(img_feat, txt_feat)
print(f'晚期融合输出: {output.shape}')

### 1.3 交叉注意力融合 (Cross-Attention)

In [ ]:
class CrossAttentionFusion(nn.Module):
    """交叉注意力融合：BLIP/Flamingo 的核心机制"""
    def __init__(self, dim=512, num_heads=8, num_layers=4):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                'self_attn': nn.MultiheadAttention(dim, num_heads, batch_first=True),
                'cross_attn': nn.MultiheadAttention(dim, num_heads, batch_first=True),
                'ffn': nn.Sequential(nn.Linear(dim, dim*4), nn.GELU(), nn.Linear(dim*4, dim)),
                'norm1': nn.LayerNorm(dim),
                'norm2': nn.LayerNorm(dim),
                'norm3': nn.LayerNorm(dim)
            }) for _ in range(num_layers)
        ])
    
    def forward(self, query, context):
        x = query
        for layer in self.layers:
            x = x + layer['self_attn'](layer['norm1'](x), layer['norm1'](x), layer['norm1'](x))[0]
            x = x + layer['cross_attn'](layer['norm2'](x), context, context)[0]
            x = x + layer['ffn'](layer['norm3'](x))
        return x

# 测试
cross_attn = CrossAttentionFusion().to(device)
query = torch.randn(4, 32, 512).to(device)   # 文本
context = torch.randn(4, 49, 512).to(device) # 图像
output = cross_attn(query, context)
print(f'交叉注意力输出: {output.shape}')

In [ ]:
# 可视化融合策略对比
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

strategies = ['早期融合', '晚期融合', '交叉注意力']
pros = ['模态交互充分', '计算高效', '灵活精细']
cons = ['计算量大', '交互有限', '复杂度高']

for i, (ax, name, pro, con) in enumerate(zip(axes, strategies, pros, cons)):
    ax.text(0.5, 0.7, name, ha='center', va='center', fontsize=14, fontweight='bold')
    ax.text(0.5, 0.45, f'优点: {pro}', ha='center', va='center', fontsize=11, color='green')
    ax.text(0.5, 0.25, f'缺点: {con}', ha='center', va='center', fontsize=11, color='red')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    ax.add_patch(plt.Rectangle((0.05, 0.05), 0.9, 0.9, fill=False, ec='black', lw=2))

plt.suptitle('多模态融合策略对比', fontsize=14)
plt.tight_layout()
plt.show()

## 2. 图文检索系统实现

In [ ]:
class ImageTextRetrieval:
    """基于 CLIP 的图文检索系统"""
    def __init__(self, embed_dim=512):
        self.embed_dim = embed_dim
        self.image_embeddings = None
        self.image_ids = []
    
    def build_index(self, image_features, image_ids):
        """构建图像索引"""
        self.image_embeddings = F.normalize(image_features, dim=-1)
        self.image_ids = image_ids
        print(f'索引构建完成: {len(image_ids)} 张图像')
    
    def search_by_text(self, text_features, top_k=5):
        """文本搜索图像"""
        text_features = F.normalize(text_features, dim=-1)
        similarity = text_features @ self.image_embeddings.T
        scores, indices = similarity.topk(top_k, dim=-1)
        return [(self.image_ids[idx], score.item()) for idx, score in zip(indices[0], scores[0])]
    
    def search_by_image(self, image_features, top_k=5):
        """以图搜图"""
        image_features = F.normalize(image_features, dim=-1)
        similarity = image_features @ self.image_embeddings.T
        scores, indices = similarity.topk(top_k, dim=-1)
        return [(self.image_ids[idx], score.item()) for idx, score in zip(indices[0], scores[0])]

# 测试检索系统
retrieval = ImageTextRetrieval()
image_features = torch.randn(100, 512)  # 100张图像
image_ids = [f'img_{i:03d}' for i in range(100)]
retrieval.build_index(image_features, image_ids)

# 文本检索
query_text = torch.randn(1, 512)
results = retrieval.search_by_text(query_text, top_k=5)
print('\n文本检索结果:')
for img_id, score in results:
    print(f'  {img_id}: {score:.4f}')

## 3. 模型优化技术

### 3.1 知识蒸馏

In [ ]:
class DistillationLoss(nn.Module):
    """知识蒸馏损失"""
    def __init__(self, temperature=4.0, alpha=0.5):
        super().__init__()
        self.temperature = temperature
        self.alpha = alpha
    
    def forward(self, student_logits, teacher_logits, labels):
        # 软标签损失
        soft_loss = F.kl_div(
            F.log_softmax(student_logits / self.temperature, dim=-1),
            F.softmax(teacher_logits / self.temperature, dim=-1),
            reduction='batchmean'
        ) * (self.temperature ** 2)
        
        # 硬标签损失
        hard_loss = F.cross_entropy(student_logits, labels)
        
        return self.alpha * soft_loss + (1 - self.alpha) * hard_loss

# 测试
distill_loss = DistillationLoss()
student_out = torch.randn(8, 100)
teacher_out = torch.randn(8, 100)
labels = torch.randint(0, 100, (8,))
loss = distill_loss(student_out, teacher_out, labels)
print(f'蒸馏损失: {loss.item():.4f}')

### 3.2 梯度检查点 (节省显存)

In [ ]:
from torch.utils.checkpoint import checkpoint

class MemoryEfficientEncoder(nn.Module):
    """使用梯度检查点的编码器"""
    def __init__(self, dim=512, num_layers=12):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(dim, nhead=8, batch_first=True)
            for _ in range(num_layers)
        ])
        self.use_checkpoint = True
    
    def forward(self, x):
        for layer in self.layers:
            if self.use_checkpoint and self.training:
                x = checkpoint(layer, x, use_reentrant=False)
            else:
                x = layer(x)
        return x

# 对比显存使用
encoder = MemoryEfficientEncoder().to(device)
x = torch.randn(4, 196, 512).to(device)

encoder.use_checkpoint = False
out1 = encoder(x)
print(f'无检查点输出: {out1.shape}')

encoder.use_checkpoint = True
out2 = encoder(x)
print(f'有检查点输出: {out2.shape}')

## 4. 训练技巧

### 4.1 学习率调度

In [ ]:
def get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps):
    """余弦退火 + 预热"""
    def lr_lambda(current_step):
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        progress = float(current_step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
        return max(0.0, 0.5 * (1.0 + np.cos(np.pi * progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# 可视化学习率曲线
model = nn.Linear(10, 10)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scheduler = get_cosine_schedule_with_warmup(optimizer, 1000, 10000)

lrs = []
for _ in range(10000):
    lrs.append(optimizer.param_groups[0]['lr'])
    scheduler.step()

plt.figure(figsize=(10, 4))
plt.plot(lrs)
plt.xlabel('Step')
plt.ylabel('Learning Rate')
plt.title('Cosine Schedule with Warmup')
plt.axvline(x=1000, color='r', linestyle='--', label='Warmup End')
plt.legend()
plt.show()

### 4.2 梯度累积

In [ ]:
def train_with_gradient_accumulation(model, dataloader, optimizer, accumulation_steps=4):
    """梯度累积训练"""
    model.train()
    optimizer.zero_grad()
    
    for i, (images, texts) in enumerate(dataloader):
        # 前向传播
        loss = model(images, texts) / accumulation_steps
        
        # 反向传播 (累积梯度)
        loss.backward()
        
        # 每 accumulation_steps 步更新一次
        if (i + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

print('梯度累积: 等效批次大小 = 实际批次 x 累积步数')
print('例如: batch_size=8, accumulation=4 -> 等效 batch_size=32')

## 5. 评估指标

In [ ]:
def compute_retrieval_metrics(similarity, k_values=[1, 5, 10]):
    """计算检索指标: Recall@K, MRR, MAP"""
    N = similarity.shape[0]
    results = {}
    
    # Recall@K
    for k in k_values:
        _, topk = similarity.topk(min(k, N), dim=-1)
        correct = torch.arange(N).unsqueeze(1)
        hits = (topk == correct).any(dim=-1).float()
        results[f'R@{k}'] = hits.mean().item() * 100
    
    # MRR (Mean Reciprocal Rank)
    ranks = (similarity.argsort(dim=-1, descending=True) == torch.arange(N).unsqueeze(1)).float().argmax(dim=-1) + 1
    results['MRR'] = (1.0 / ranks.float()).mean().item() * 100
    
    return results

# 测试
sim = torch.randn(50, 50)
sim[torch.arange(50), torch.arange(50)] += 2  # 增加对角线相似度
metrics = compute_retrieval_metrics(sim)
print('检索指标:')
for k, v in metrics.items():
    print(f'  {k}: {v:.2f}%')

## 总结

| 主题 | 关键技术 | 应用场景 |
|:-----|:---------|:---------|
| 融合策略 | 早期/晚期/交叉注意力 | 模型设计 |
| 检索系统 | 向量索引 + 相似度搜索 | 图文检索 |
| 模型优化 | 蒸馏/量化/检查点 | 部署加速 |
| 训练技巧 | 学习率调度/梯度累积 | 训练稳定 |